# Chained DiD for incomplete panels

This tutorial constructs an overlapping panel with known treatment effects of 3 and 7. Each unit is observed twice, and no unit spans the reference date and final date. The experimental native estimator combines the observed changes under PT-post and the stated observation assumptions.

The example is a numerical illustration. It does not establish coverage, validate a real missingness process or reproduce the full R `cdid` feature set.
Pairs need two observed units in each arm; excluded candidates and low-support warnings are recorded in `result.diagnostics`. Identity GMM ignores pair precision, and sparse-pair intervals can remain anti-conservative. The warning threshold of 20 is not a qualification rule. Mixed diagnostic horizons exclude structural reference identities. Use `result.aggregation_weights("simple")` to inspect constituent weights.


In [ ]:
import numpy as np
import pandas as pd
from diff_diff import ChainedDiD, plot_event_study

## Construct the overlapping windows

Cohort 2 starts treatment at date 2; date 1 is its reference. The untreated trend has a fixed earlier cohort difference, so PT-post does not require a zero earlier diagnostic. Within each observation window the constructed slope deviations average to zero.

In [ ]:
rows = []
for cohort in (0, 2):
    for s, t in ((0, 1), (1, 2), (2, 3)):
        for j, slope in enumerate((-1.5, -0.5, 0.5, 1.5)):
            for date in (s, t):
                effect = {2: 3, 3: 7}.get(date, 0) if cohort else 0
                y = j + (3 + slope) * date
                if cohort:
                    y += 10 + 2 * (date >= 1) + effect
                rows.append((f"{cohort}-{s}-{j}", date, cohort, y))
panel = pd.DataFrame(rows, columns=["id", "date", "cohort", "outcome"])
assert panel.groupby("id").size().eq(2).all()
panel.head()

## Fit and inspect the original targets

Without covariates the pairwise IPW comparison reduces to a difference in mean changes. All supported pairs enter the identity-GMM system. The headline weights scheduled post-treatment cells by cohort size; inference uses unit HC0 covariance and normal pointwise intervals.

In [ ]:
result = ChainedDiD().fit(
    panel, outcome="outcome", unit="id", time="date", first_treat="cohort"
)
assert np.isclose(result.group_time_effects[2, 2].effect, 3)
assert np.isclose(result.group_time_effects[2, 3].effect, 7)
assert np.isclose(result.group_time_effects[2, 0].effect, -2)
print(result.summary())
result.to_dataframe()
weights = result.aggregation_weights("simple")
assert np.allclose(weights.weight, [0.5, 0.5])
weights


In [ ]:
event = result.aggregate("event_study")
assert np.allclose(event.att[event.event_time >= 0], [3, 7])
plot_event_study(event, show=False)
event.to_dataframe()

## Remove a required bridge

Keep the calendar rows but make date-2 outcomes unobserved. There is then no supported path from the reference date to either post-treatment target. The original summary becomes unavailable; the estimator does not replace it with an average over a different set of effects.

In [ ]:
broken = panel.copy()
broken.loc[broken.date == 2, "outcome"] = np.nan
unsupported = ChainedDiD().fit(broken, "outcome", "id", "date", "cohort")
assert np.isnan(unsupported.att)
assert unsupported.group_time_effects[2, 3].skip_reason == "unsupported_target"
unsupported.to_dataframe()[["group", "time", "att", "available", "status"]]

## Conditions and limits

For covariate adjustment, supply complete static numeric columns through `covariates=[...]`. A sufficient baseline observation condition preserves the joint covariate and actual-change distribution within each cohort for every used pair. Treatment IPW alone does not correct conditional attrition.

A unit-constant `cluster` column selects one-way CR1 covariance. Normal intervals do not guarantee few-cluster validity. Units with fewer than two observed outcomes remain in population accounting when represented in the input. Public bootstrap intervals, PT-all, not-yet-treated controls, OR/DR and survey designs are outside this implementation.

Reference: Bellégo, Benatia and Dortet-Bernadet, [The chained difference-in-differences](https://doi.org/10.1016/j.jeconom.2024.105783). The development validation matrix and source-hashed simulation manifest distinguish executed evidence from unrun reference gates.